# Notebook 3: Preprocessing Pipeline
## Feature Engineering, Custom Transformers, TF-IDF, Pipeline Assembly

**Key Steps**:
1. Handle missing values
2. Custom TextFeatureExtractor transformer (high-marks component)
3. Custom SummaryFeatureExtractor transformer
4. NLP pipeline: Tokenizer → StopWordsRemover → TF-IDF
5. Feature assembly and scaling
6. Save preprocessing pipeline and transformed data

In [ ]:
# ============================================================================
# Cell 1: Setup
# ============================================================================
import os, sys, time
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover, HashingTF, IDF,
    StringIndexer, VectorAssembler, StandardScaler, SQLTransformer
)

# Import custom transformers
from scripts.custom_transformer import TextFeatureExtractor, SummaryFeatureExtractor

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ============================================================================
# Cell 2: Create Spark Session
# ============================================================================
spark = (
    SparkSession.builder
    .appName('RedditVirality_Preprocessing')
    .master('local[*]')
    .config('spark.driver.memory', '8g')
    .config('spark.executor.memory', '4g')
    .config('spark.driver.maxResultSize', '4g')
    .config('spark.sql.shuffle.partitions', 200)
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready')

In [ ]:
# ============================================================================
# Cell 3: Load Parquet Data (from Notebook 2 output)
# ============================================================================
PARQUET_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'reddit_posts.parquet')
SAMPLE_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'sample', 'reddit_sample.parquet')

# Use sample for development; switch to full for production
USE_SAMPLE = True  # Set to False for full dataset

if USE_SAMPLE and os.path.exists(SAMPLE_PATH):
    df = spark.read.parquet(SAMPLE_PATH)
    print(f'Loaded SAMPLE data: {df.count():,} rows')
else:
    df = spark.read.parquet(PARQUET_PATH)
    print(f'Loaded FULL data: {df.count():,} rows')

df.printSchema()
print(f'\nClass distribution:')
df.groupBy('is_viral').count().show()

## Step 1: Handle Missing Values

In [ ]:
# ============================================================================
# Cell 4: Missing Value Handling
# ============================================================================
print('MISSING VALUE HANDLING')
print('=' * 60)

total = df.count()
for col_name in df.columns:
    null_ct = df.filter(F.col(col_name).isNull()).count()
    print(f'  {col_name:20s}: {null_ct:>8,} nulls ({null_ct/total*100:.2f}%)')

# Fill missing text columns with empty string
text_cols = ['body', 'normalizedBody', 'content', 'summary', 'author']
for col_name in text_cols:
    if col_name in df.columns:
        df = df.withColumn(col_name, F.coalesce(F.col(col_name), F.lit('')))

# Drop rows still missing critical data
df = df.filter(
    (F.length('body') > 10) &  # At least 10 chars in body
    (F.col('subreddit').isNotNull())
)

print(f'\nRows after cleaning: {df.count():,}')

## Step 2: Custom Transformer — TextFeatureExtractor

This is a **custom PySpark ML Transformer** that extracts hand-crafted features from Reddit post text. This is crucial for demonstrating advanced pipeline integration.

In [ ]:
# ============================================================================
# Cell 5: Apply Custom TextFeatureExtractor
# ============================================================================
print('CUSTOM TRANSFORMER: TextFeatureExtractor')
print('=' * 60)

# Apply the custom transformer
text_extractor = TextFeatureExtractor(inputCol='normalizedBody', outputCol='text_feats')
df_text = text_extractor.transform(df)

# Show extracted features
feature_cols = ['char_count', 'word_count', 'sentence_count', 'avg_word_length',
                'question_count', 'exclamation_count', 'uppercase_ratio', 
                'has_url', 'paragraph_count']

print('\nExtracted text features (sample):')
df_text.select(feature_cols).describe().show()

print('Feature descriptions:')
for feat in feature_cols:
    print(f'  • {feat}')

print(f'\n✓ TextFeatureExtractor produced {len(feature_cols)} features')

In [ ]:
# ============================================================================
# Cell 6: Apply Custom SummaryFeatureExtractor
# ============================================================================
print('CUSTOM TRANSFORMER: SummaryFeatureExtractor')
print('=' * 60)

summary_extractor = SummaryFeatureExtractor(inputCol='summary', outputCol='summ_feats')
df_summ = summary_extractor.transform(df_text)

summary_features = ['summary_length', 'summary_word_count', 'content_density']
print('\nSummary features (sample):')
df_summ.select(summary_features).describe().show()

print(f'✓ SummaryFeatureExtractor produced {len(summary_features)} features')

In [ ]:
# ============================================================================
# Cell 7: Compute Unique Word Ratio
# ============================================================================
df_enriched = df_summ.withColumn(
    'unique_word_ratio',
    F.when(F.col('word_count') > 0,
           F.size(F.array_distinct(F.split(F.col('normalizedBody'), ' '))).cast('float') / F.col('word_count'))
    .otherwise(0.0).cast('float')
)

print('Unique word ratio statistics:')
df_enriched.select('unique_word_ratio').describe().show()

## Step 3: NLP Pipeline — Tokenizer → StopWords → TF-IDF

In [ ]:
# ============================================================================
# Cell 8: Build Complete ML Preprocessing Pipeline
# ============================================================================
print('BUILDING PREPROCESSING PIPELINE')
print('=' * 60)

# Stage 1: Encode subreddit
subreddit_indexer = StringIndexer(
    inputCol='subreddit', outputCol='subreddit_index', handleInvalid='keep'
)

# Stage 2: Tokenizer
tokenizer = Tokenizer(inputCol='normalizedBody', outputCol='words')

# Stage 3: Stop words removal
stopwords_remover = StopWordsRemover(inputCol='words', outputCol='filtered_words')

# Stage 4: Hashing TF
hashing_tf = HashingTF(
    inputCol='filtered_words', outputCol='raw_tfidf', numFeatures=5000
)

# Stage 5: IDF
idf = IDF(inputCol='raw_tfidf', outputCol='tfidf_features', minDocFreq=5)

# Stage 6: Assemble all features
numeric_features = [
    'char_count', 'word_count', 'sentence_count', 'avg_word_length',
    'unique_word_ratio', 'question_count', 'exclamation_count',
    'uppercase_ratio', 'has_url', 'paragraph_count',
    'summary_length', 'summary_word_count', 'content_density'
]

assembler = VectorAssembler(
    inputCols=numeric_features + ['subreddit_index', 'tfidf_features'],
    outputCol='raw_features',
    handleInvalid='skip'
)

# Stage 7: Standard scaling
scaler = StandardScaler(
    inputCol='raw_features', outputCol='features',
    withStd=True, withMean=False  # withMean=False for sparse vectors
)

# Create pipeline (only the ML stages; custom transformers already applied)
pipeline = Pipeline(stages=[
    subreddit_indexer, tokenizer, stopwords_remover,
    hashing_tf, idf, assembler, scaler
])

print('Pipeline stages:')
for i, stage in enumerate(pipeline.getStages()):
    print(f'  {i+1}. {stage.__class__.__name__}')

print(f'\nTotal numeric features: {len(numeric_features)}')
print(f'TF-IDF features: 5000')
print(f'Total feature dimensions: {len(numeric_features) + 1 + 5000}')

In [ ]:
# ============================================================================
# Cell 9: Fit and Transform
# ============================================================================
start_time = time.time()

# Fit pipeline
pipeline_model = pipeline.fit(df_enriched)
fit_time = time.time() - start_time
print(f'Pipeline fit time: {fit_time:.1f}s')

# Transform data
start_time = time.time()
df_transformed = pipeline_model.transform(df_enriched)
transform_time = time.time() - start_time
print(f'Pipeline transform time: {transform_time:.1f}s')

# Select only needed columns for modeling
df_final = df_transformed.select('id', 'features', 'is_viral', 'subreddit')
print(f'\nFinal schema:')
df_final.printSchema()
print(f'Rows: {df_final.count():,}')

In [ ]:
# ============================================================================
# Cell 10: Train-Test Split
# ============================================================================
train_df, test_df = df_final.randomSplit([0.8, 0.2], seed=42)

print('TRAIN-TEST SPLIT')
print('=' * 60)
print(f'Training set: {train_df.count():,} rows')
print(f'Test set:     {test_df.count():,} rows')

print('\nTraining class distribution:')
train_df.groupBy('is_viral').count().show()

print('Test class distribution:')
test_df.groupBy('is_viral').count().show()

In [ ]:
# ============================================================================
# Cell 11: Save Pipeline Model and Processed Data
# ============================================================================
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# Save fitted pipeline
pipeline_path = os.path.join(MODEL_DIR, 'preprocessing_pipeline')
pipeline_model.write().overwrite().save(pipeline_path)
print(f'Pipeline model saved to: {pipeline_path}')

# Save train/test sets
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

train_df.write.mode('overwrite').parquet(os.path.join(PROCESSED_DIR, 'train.parquet'))
test_df.write.mode('overwrite').parquet(os.path.join(PROCESSED_DIR, 'test.parquet'))

print(f'Training data saved to: {os.path.join(PROCESSED_DIR, "train.parquet")}')
print(f'Test data saved to: {os.path.join(PROCESSED_DIR, "test.parquet")}')

# Also save full transformed data with all features for Tableau
df_transformed.select(
    'id', 'subreddit', 'is_viral',
    *numeric_features, 'subreddit_index'
).write.mode('overwrite').parquet(os.path.join(PROCESSED_DIR, 'features_all.parquet'))

print('\n✓ All preprocessing artifacts saved successfully')

In [ ]:
# ============================================================================
# Cell 12: Preprocessing Summary
# ============================================================================
print('=' * 60)
print('PREPROCESSING PIPELINE SUMMARY')
print('=' * 60)
print(f'Custom transformers:   TextFeatureExtractor, SummaryFeatureExtractor')
print(f'Numeric features:      {len(numeric_features)}')
print(f'  - Text features:     10 (char_count, word_count, etc.)')
print(f'  - Summary features:  3 (summary_length, word_count, density)')
print(f'Categorical features:  1 (subreddit_index)')
print(f'TF-IDF dimensions:     5,000')
print(f'Total feature vector:  {len(numeric_features) + 1 + 5000} dimensions')
print(f'Pipeline stages:       7')
print(f'Pipeline fit time:     {fit_time:.1f}s')
print(f'Training rows:         {train_df.count():,}')
print(f'Test rows:             {test_df.count():,}')
print('=' * 60)

spark.stop()
print('Spark session stopped.')